<a href="https://colab.research.google.com/github/quinteleg/An-lisis_y_Visualizacion_de_Datos_para_la_Toma_de_Decisiones/blob/main/clases/semana02-clase04-python-datos-preparacion-sql/notebooks/lab-python-datos-preparacion-sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio: Python aplicado a datos y preparación para SQL
### Semana 2 · Clase 4
**Curso:** Análisis y Visualización de Datos para la Toma de Decisiones (Esumer)

La clase pasada aprendimos lo básico de Python usando datos que escribíamos directamente en el código. Hoy damos el siguiente paso: traer datos reales desde un archivo (CSV o Excel), explorarlos y hacerles preguntas de negocio con operaciones simples. Esta clase es también el puente hacia la Clase 5, donde empezamos a usar SQL: muchas de las cosas que hagamos hoy en Python (filtrar, contar, resumir) son exactamente lo que en un momento vamos a pedirle a una base de datos.

Para leer y explorar los archivos vamos a usar **Pandas**, la librería de Python más usada para trabajar con datos en forma de tabla. Hoy la usamos solo para lo esencial (cargar un archivo y explorarlo); en el Módulo 2 (a partir de la Clase 7) la vamos a estudiar a fondo.

Ejecuta las celdas en orden, de arriba hacia abajo (▶ o `Shift + Enter`).

## 1. Subir un archivo a Google Colab

Antes de leer un archivo con Python, ese archivo tiene que existir dentro del entorno donde corre Colab (que vive en la nube, no en tu computador). La forma más directa de llevar un archivo desde tu computador hasta Colab es subiéndolo con la utilidad `files.upload()`, de la librería propia de Colab.

Antes de continuar, descarga a tu computador estos dos archivos desde la carpeta `datos/` de esta misma clase, en el repositorio del curso: `ventas_tienda.csv` y `ventas_tienda.xlsx`. Los vamos a usar durante toda la clase.

In [ ]:
from google.colab import files

uploaded = files.upload()

Al ejecutar la celda anterior aparece un botón para elegir un archivo de tu computador. Selecciona `ventas_tienda.csv` (puedes subir varios archivos a la vez, seleccionándolos todos). Cuando termine de subir, Colab confirma el nombre y el tamaño del archivo.

El archivo queda guardado en la carpeta de trabajo del entorno de Colab, que se llama `/content/`. Para confirmarlo, podemos listar los archivos que hay ahí.

In [ ]:
!ls -la /content/

Deberías ver `ventas_tienda.csv` en la lista. Ahora sube también `ventas_tienda.xlsx`, de la misma manera: ejecuta de nuevo `files.upload()` y selecciónalo.

In [ ]:
uploaded = files.upload()

Un detalle importante: el entorno de Colab es **efímero**. Si cierras la pestaña, dejas mucho tiempo inactivo el notebook, o le das a "Entorno de ejecución → Reiniciar", los archivos que subiste con `files.upload()` **se borran** y hay que volver a subirlos. Esto es distinto a lo que pasa con el notebook mismo (que si lo guardas en tu Drive, no se pierde): lo efímero es el archivo de datos que subiste a esa sesión, no el notebook.

Si vas a trabajar con los mismos archivos muchas veces y no quieres subirlos cada vez, la alternativa es **montar tu Google Drive** dentro de Colab, para que el archivo quede disponible en todas tus sesiones. Por ahora, para esta clase, con `files.upload()` es suficiente; solo queda como referencia si más adelante te hace falta:

```python
from google.colab import drive
drive.mount('/content/drive')
```

## 2. Leer el archivo con pandas

Para leer un archivo CSV, la función es `pandas.read_csv()`: le pasas la ruta del archivo (basta con el nombre, porque ya está en `/content/`) y te devuelve un **DataFrame**, que es la forma que tiene pandas de representar una tabla: filas y columnas, muy parecido a la lista de diccionarios que usamos en la Clase 3, pero con muchas más herramientas ya construidas encima. Vamos a explorar esa estructura a fondo en el Módulo 2; por ahora la usamos simplemente como "la tabla que acabo de cargar".

In [ ]:
import pandas as pd

df = pd.read_csv("ventas_tienda.csv")
df.head()

`df.head()` muestra las primeras 5 filas, para darte una idea rápida de cómo se ve la tabla sin imprimirla completa. También podemos revisar cuántas filas y columnas tiene, con `.shape` (el mismo tipo de información que ya pedíamos con `len()` sobre una lista, pero ahora en dos dimensiones: filas y columnas).

In [ ]:
df.shape

Para un archivo de Excel, la función equivalente es `pandas.read_excel()`. Funciona igual: se le pasa la ruta del archivo y devuelve un DataFrame.

In [ ]:
df_excel = pd.read_excel("ventas_tienda.xlsx")
df_excel.head()

Actividad: confirma que `df_excel` tiene las mismas filas y columnas que `df` (los dos archivos traen los mismos datos, uno en CSV y otro en Excel). Compara sus `.shape`.

In [ ]:
# TODO: compara df.shape y df_excel.shape


## 3. Problemas típicos al leer un archivo

Leer un archivo casi nunca es tan limpio como el ejemplo anterior. Los tres problemas más comunes, sobre todo con archivos que vienen de Excel en español, son el separador de columnas, la codificación de caracteres y el símbolo de decimales. Para verlos de verdad, sube ahora un tercer archivo: `ventas_tienda_excel_es.csv` (también está en `datos/`), que trae exactamente estos problemas a propósito.

In [ ]:
uploaded = files.upload()

**Problema 1: la codificación de caracteres.** Por defecto, `read_csv()` espera que el archivo esté guardado en UTF-8 (la codificación más común hoy en día). Muchos archivos exportados desde Excel en Windows quedan guardados en otra codificación (Latin-1), y cualquier tilde o eñe rompe la lectura.

In [ ]:
# Ejecuta esta celda para ver el error real de un problema de codificación
df_es = pd.read_csv("ventas_tienda_excel_es.csv")

El error `UnicodeDecodeError` dice que un byte del archivo no se pudo interpretar como UTF-8. La solución es indicarle a `read_csv()` cuál es la codificación real del archivo, con el parámetro `encoding`. Para estos casos, casi siempre funciona `"latin-1"` (también llamada `"ISO-8859-1"`).

In [ ]:
df_es = pd.read_csv("ventas_tienda_excel_es.csv", encoding="latin-1")
df_es.head()

**Problema 2: el separador de columnas.** Fíjate en el resultado anterior: aunque ya no dio error, algo sigue mal. `pandas.read_csv()` asume por defecto que las columnas están separadas por comas (`,`). Pero Excel configurado en español separa las columnas con punto y coma (`;`), porque reserva la coma para los decimales. Si no coincide el separador, pandas cree que toda la fila es una sola columna gigante.

In [ ]:
df_es.shape

En vez de 8 filas y 7 columnas, quedó todo comprimido en 1 sola columna. Se soluciona indicando el separador real con el parámetro `sep`.

In [ ]:
df_es = pd.read_csv("ventas_tienda_excel_es.csv", encoding="latin-1", sep=";")
df_es.head()

In [ ]:
df_es.shape

Ahora sí: 8 filas y 7 columnas, con las tildes y las eñes bien escritas. Dos notas adicionales que vale la pena tener en cuenta:

- Los nombres de columnas de este archivo tienen espacios, como `"Precio Unitario"`. Cuando un nombre de columna tiene espacios, hay que acceder a ella con corchetes y comillas, `df_es["Precio Unitario"]`, porque la forma corta con punto (`df_es.Precio_Unitario`) no funciona con espacios.
- Fíjate también en la columna `"Precio Unitario"`: los valores se ven como números (`8500,00`), pero como usan coma para el decimal, pandas los interpretó como texto en vez de como números. Ese es justamente el tema de la siguiente sección: cómo revisar y corregir el tipo de dato de una columna.

Actividad: el archivo `ventas_tienda_excel_es.csv` que ya subiste tiene los mismos tres problemas. Sin mirar el código de arriba, vuelve a leerlo desde cero en una variable `df_reto`, indicando tú mismo el `encoding` y el `sep` correctos, y confirma con `.shape` que te queda con 8 filas y 7 columnas.

In [ ]:
# TODO: lee ventas_tienda_excel_es.csv en df_reto con el encoding y el separador correctos
df_reto = None



## 4. Exploración de columnas y tipos de datos

Antes de operar sobre una tabla conviene mirarla con cuidado: qué columnas tiene y qué tipo de dato trae cada una. `df.columns` devuelve la lista de nombres de columnas, y `df.dtypes` devuelve el tipo de dato de cada una (una por una). También existe `df.info()`, que muestra ambas cosas juntas, más el número de filas.

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
df.info()

### Los tipos de datos en pandas

Pandas usa sus propios nombres para los tipos de dato, parecidos a los de Python que ya conoces de la Clase 3, pero no idénticos:

| Tipo en pandas | Qué guarda | Equivalente de Python que ya conoces |
|---|---|---|
| `int64` | Números enteros | `int` |
| `float64` | Números decimales | `float` |
| `str` | Texto | `str` |
| `object` | Una columna con una mezcla de tipos (no es puramente texto ni puramente número) | (no tiene un equivalente directo) |
| `bool` | Verdadero / falso | `bool` |
| `datetime64` | Fechas y horas | (no lo vimos en Python puro) |

El caso que más te vas a encontrar es una columna que "se ve" como números pero pandas la marca como `str`: eso casi siempre es una señal de que hay que revisarla y convertirla, porque algún detalle del archivo (una coma decimal, un símbolo como `$`, un espacio de más) le impidió a pandas reconocerla como número.

Mira el resultado de `df.dtypes` de hace un momento: la columna `fecha` aparece como `str` (texto), aunque claramente representa una fecha. Esto es normal: pandas no adivina que algo es una fecha solo por su forma, hay que decírselo explícitamente.

### Cómo cambiar el tipo de una columna (casting)

Cambiar el tipo de una columna después de leerla se llama comúnmente "castear". La herramienta más general es el método `.astype()`, que sirve para tipos simples (`int`, `float`, `str`). Para fechas, en cambio, se usa una función aparte: `pd.to_datetime()`, que entiende varios formatos de fecha automáticamente.

In [ ]:
df["fecha"] = pd.to_datetime(df["fecha"])
df.dtypes

Ahora `fecha` aparece como `datetime64` (un tipo de fecha real). Al quedar como fecha de verdad (y no como texto), pandas nos deja hacer cosas que con texto no se pueden, como comparar fechas o calcular cuántos días pasaron entre dos.

Ahora vamos a corregir el problema de la columna `"Precio Unitario"` del archivo en español, que quedó como texto por la coma decimal. Si intentamos convertirla directamente con `.astype(float)`, falla.

In [ ]:
# Ejecuta esta celda para ver el error real de intentar castear un texto con coma decimal
df_es["Precio Unitario"].astype(float)

El error `ValueError: could not convert string to float` es porque Python no reconoce la coma como separador decimal (para Python y pandas, el separador decimal siempre es el punto). La solución es reemplazar la coma por un punto en el texto, y solo después convertir a número, con `.str.replace()` encadenado con `.astype(float)`.

In [ ]:
df_es["Precio Unitario"] = df_es["Precio Unitario"].str.replace(",", ".", regex=False).astype(float)
df_es.dtypes

Cuando el problema no es el formato sino que de verdad hay valores inválidos mezclados en la columna (por ejemplo, texto como `"sin dato"` en medio de una columna numérica), forzar con `.astype()` directamente rompe todo el proceso. Para esos casos existe `pd.to_numeric()`, con el parámetro `errors="coerce"`: convierte lo que sí es un número, y lo que no, lo reemplaza por `NaN` (el valor que pandas usa para "dato faltante"), en vez de detener todo con un error.

In [ ]:
precios_con_error = pd.Series(["8500", "320000", "sin dato", "1500"])

# Con astype fallaría; con to_numeric y errors="coerce" no se detiene
pd.to_numeric(precios_con_error, errors="coerce")

Actividad: en `df_reto` (el DataFrame que armaste en la actividad anterior), castea la columna `"Fecha"` a fecha con `pd.to_datetime()`, y la columna `"Precio Unitario"` a número decimal (recuerda que trae coma como separador de decimales). Confirma el resultado con `df_reto.dtypes`.

In [ ]:
# TODO: castea Fecha a datetime y Precio Unitario a float en df_reto



## 5. Operaciones básicas

Una de las ventajas más grandes de pandas frente a recorrer los datos a mano con un `for` (como hicimos en la Clase 3) es que las operaciones sobre una columna completa se aplican a **todas las filas a la vez**, sin escribir ningún ciclo. Por ejemplo, para calcular el total de cada venta (cantidad por precio unitario), basta con multiplicar las dos columnas directamente.

In [ ]:
df["total"] = df["cantidad"] * df["precio_unitario"]
df.head()

`df["cantidad"] * df["precio_unitario"]` multiplicó cada valor de `cantidad` por el valor de `precio_unitario` de esa misma fila, para las 20 filas de una sola vez, y `df["total"] = ...` guardó ese resultado como una columna nueva. Esto es lo mismo que en la Clase 3 hacíamos con un `for` recorriendo cada registro uno por uno, pero mucho más directo.

También se puede operar con un solo número: por ejemplo, para calcular cuánto sería cada venta con un descuento del 10%.

In [ ]:
df["total_con_descuento"] = df["total"] * 0.9
df[["producto", "total", "total_con_descuento"]].head()

Actividad: crea una columna nueva `iva` que sea el 19% del `total` de cada venta (`df["total"] * 0.19`), y otra columna `total_con_iva` que sea `total + iva`. Muestra las primeras filas de `producto`, `total`, `iva` y `total_con_iva`.

In [ ]:
# TODO: crea las columnas iva y total_con_iva



## 6. Filtros

Filtrar una tabla significa quedarte solo con las filas que cumplen una condición, exactamente la idea de un `WHERE` en SQL (que veremos en la Clase 5). En pandas se hace escribiendo la condición dentro de corchetes: `df[condicion]`. Por dentro, `df["categoria"] == "Tecnologia"` genera una columna de `True`/`False`, una por fila, y `df[esa_columna]` se queda solo con las filas marcadas `True`.

In [ ]:
ventas_tecnologia = df[df["categoria"] == "Tecnologia"]
ventas_tecnologia

Para combinar más de una condición, se usan `&` (y) y `|` (o), no las palabras `and`/`or` que usamos en la Clase 3: en pandas, `and`/`or` no funcionan fila por fila, así que hay que usar `&`/`|`, y cada condición debe ir entre paréntesis.

In [ ]:
ventas_medellin_altas = df[(df["ciudad"] == "Medellin") & (df["cantidad"] > 5)]
ventas_medellin_altas

Actividad: filtra `df` para quedarte solo con las ventas donde el `vendedor` sea `"Ana"` **o** el `producto` sea `"Mochila"`. Guarda el resultado en `ventas_filtradas` y muéstralo.

In [ ]:
# TODO: filtra por vendedor == "Ana" o producto == "Mochila"
ventas_filtradas = None



## 7. Conteos y resúmenes

Para contar cuántas veces aparece cada valor de una columna categórica, existe `.value_counts()`. Y para resumir una columna numérica, están los mismos nombres que ya conoces de la Clase 3: `.sum()`, `.mean()`, `.max()`, `.min()`, ahora aplicados sobre una columna completa de un DataFrame en vez de sobre una lista.

In [ ]:
df["producto"].value_counts()

In [ ]:
print("Total vendido:", df["total"].sum())
print("Venta promedio:", df["total"].mean())
print("Venta más alta:", df["total"].max())
print("Venta más baja:", df["total"].min())

Estas dos ideas (filtrar y resumir) se combinan todo el tiempo: primero te quedas con las filas que te interesan, y después resumes esa parte. Por ejemplo, el total vendido solo en Bogotá.

In [ ]:
total_bogota = df[df["ciudad"] == "Bogota"]["total"].sum()
print("Total vendido en Bogotá:", total_bogota)

Una pregunta de negocio muy común es "¿cuánto vendió cada vendedor?", es decir, un resumen por categoría. Formalmente, esto se llama agrupar (`groupby`), y lo vamos a ver a fondo en la Clase 9. Por ahora, con lo que ya sabemos (`.unique()` para obtener los valores distintos de una columna, filtrar y sumar), lo podemos resolver igual, solo que a mano, categoría por categoría.

In [ ]:
for vendedor in df["vendedor"].unique():
    total_vendedor = df[df["vendedor"] == vendedor]["total"].sum()
    print(vendedor, "vendió en total:", total_vendedor)

Actividad: calcula, para cada `categoria` de producto, cuántas ventas tuvo (`value_counts()` sirve directamente) y el total vendido en esa categoría (con el mismo patrón de `for` + filtro + `.sum()` de arriba).

In [ ]:
# TODO: cuenta las ventas por categoria y suma el total por categoria



## 8. Pensamiento tabular y lógica consulta–resultado

Todo lo que hicimos hoy tiene un paralelo directo con SQL, el lenguaje de consulta a bases de datos que empezamos en la Clase 5. La idea central es siempre la misma: partes de una tabla completa y le haces una pregunta (una consulta), y el resultado es otra tabla, más pequeña o más resumida.

| Lo que hicimos en Python (pandas) | Pregunta de negocio | Su idea equivalente en SQL |
|---|---|---|
| `df[["producto", "total"]]` | ¿Qué columnas necesito ver? | `SELECT producto, total` |
| `df[df["ciudad"] == "Medellin"]` | ¿Qué filas cumplen una condición? | `WHERE ciudad = 'Medellin'` |
| `df["producto"].value_counts()` | ¿Cuántas veces aparece cada valor? | `COUNT(*) ... GROUP BY producto` |
| `for vendedor in df["vendedor"].unique(): ...sum()` | ¿Cuánto suma cada categoría? | `SUM(total) ... GROUP BY vendedor` |

No necesitas memorizar SQL todavía, apenas es la Clase 5. La idea de esta sección es que cuando lleguemos allá, el concepto ya no sea nuevo: "filtrar", "contar" y "resumir por categoría" son la misma lógica, muestra distinto lenguaje.

## 9. Reto: responder preguntas de negocio sobre el dataset

Vuelve a partir de `df` (el DataFrame completo de `ventas_tienda.csv`, con las columnas `total`, `total_con_descuento`, `iva` y `total_con_iva` que ya construiste). Usa lo que aprendiste hoy para responder cada pregunta.

**Pregunta 1:** ¿cuál fue el total vendido de la tienda en todo el periodo?

In [ ]:
# TODO: responde la pregunta 1


**Pregunta 2:** ¿cuál fue el producto que más veces se vendió (no el que más dinero generó, sino el que más aparece en los registros)?

In [ ]:
# TODO: responde la pregunta 2


**Pregunta 3:** ¿cuántas ventas se hicieron en Cali con una cantidad mayor a 5 unidades?

In [ ]:
# TODO: responde la pregunta 3


**Pregunta 4:** ¿cuál vendedor tuvo el mayor total vendido, y cuánto fue?

In [ ]:
# TODO: responde la pregunta 4


**Pregunta 5 (reto extra):** convierte la columna `fecha` a tipo fecha si aún no lo has hecho, y responde: ¿cuántos días de ventas hay registrados entre la primera y la última fecha del dataset? (pista: `df["fecha"].max() - df["fecha"].min()`).

In [ ]:
# TODO: responde la pregunta 5 (reto extra)


## Cierre

¿Qué te costó más hoy: subir y leer el archivo, corregir los problemas de lectura, o construir los filtros y resúmenes? Guarda una copia de este notebook en tu Google Drive antes de cerrar Colab.

En la Clase 5 empezamos con bases de datos relacionales y SQL: vas a reconocer varias de las ideas de hoy (filtrar, contar, agrupar) con un lenguaje nuevo. Y en la Clase 7 retomamos pandas para estudiar a fondo el DataFrame: diagnóstico de calidad de datos, nulos, duplicados y mucho más de lo que hoy solo usamos de pasada.